In [21]:
import requests
import pandas as pd
import time
from geopy.geocoders import Nominatim

# Initialize geocoder
geolocator = Nominatim(user_agent="sg_poi_project")

def get_place_name(lat, lon):
    try:
        location = geolocator.reverse((lat, lon), timeout=10)
        if location and location.raw.get("address"):
            address = location.raw["address"]

            return (
                address.get("attraction") or
                address.get("building") or
                address.get("amenity") or
                address.get("road") or
                "Unknown"
            )
    except:
        return "Unknown"


def onemap_search_bulk_with_names(search_term, max_rows=1000):
    url = "https://www.onemap.gov.sg/api/common/elastic/search"

    all_results = []
    page = 1

    while len(all_results) < max_rows:
        params = {
            "searchVal": search_term,
            "returnGeom": "Y",
            "getAddrDetails": "Y",
            "pageNum": page
        }

        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"Error: {e}")
            break

        results = data.get("results", [])

        if not results:
            break

        for item in results:
            try:
                lat = float(item.get("LATITUDE", 0))
                lon = float(item.get("LONGITUDE", 0))
            except ValueError:
                lat, lon = None, None

            # Get place name (reverse geocode)
            place_name = get_place_name(lat, lon) if lat and lon else "Unknown"

            all_results.append({
                "Place Name": place_name,
                "address": item.get("ADDRESS"),
                "postal": item.get("POSTAL"),
                "lat": lat,
                "lon": lon,
                "building": item.get("BUILDING"),
                "block": item.get("BLK_NO"),
                "road": item.get("ROAD_NAME"),
                "source": "OneMap"
            })

            print(f"{len(all_results)} / {max_rows} processed")

            if len(all_results) >= max_rows:
                break

            # ⚠️ REQUIRED for Nominatim (avoid getting blocked)
            time.sleep(1)

        page += 1
        time.sleep(0.2)

    return pd.DataFrame(all_results)


# ✅ RUN EVERYTHING
if __name__ == "__main__":
    df = onemap_search_bulk_with_names("Punggol", max_rows=1000)

    df.to_csv("onemap_1000_with_place_names.csv", index=False)

    print(f"\n✅ Done! Saved {len(df)} rows to CSV")

1 / 1000 processed
2 / 1000 processed
3 / 1000 processed
4 / 1000 processed
5 / 1000 processed
6 / 1000 processed
7 / 1000 processed
8 / 1000 processed
9 / 1000 processed
10 / 1000 processed
11 / 1000 processed
12 / 1000 processed
13 / 1000 processed
14 / 1000 processed
15 / 1000 processed
16 / 1000 processed
17 / 1000 processed
18 / 1000 processed
19 / 1000 processed
20 / 1000 processed
21 / 1000 processed
22 / 1000 processed
23 / 1000 processed
24 / 1000 processed
25 / 1000 processed
26 / 1000 processed
27 / 1000 processed
28 / 1000 processed
29 / 1000 processed
30 / 1000 processed
31 / 1000 processed
32 / 1000 processed
33 / 1000 processed
34 / 1000 processed
35 / 1000 processed
36 / 1000 processed
37 / 1000 processed
38 / 1000 processed
39 / 1000 processed
40 / 1000 processed
41 / 1000 processed
42 / 1000 processed
43 / 1000 processed
44 / 1000 processed
45 / 1000 processed
46 / 1000 processed
47 / 1000 processed
48 / 1000 processed
49 / 1000 processed
50 / 1000 processed
51 / 1000

In [22]:
import pandas as pd
df=pd.read_csv("onemap_1000_with_place_names.csv")
df.head()

,Place Name,address,postal,lat,lon,building,block,road,source
0,Punggol Place,70A PUNGGOL PLACE PUNGGOL BUS INTERCHANGE SING...,828865,1.404278,103.902304,PUNGGOL BUS INTERCHANGE,70A,PUNGGOL PLACE,OneMap
1,Campus Boulevard,1 SENTUL WALK PUNGGOL COAST BUS INTERCHANGE SI...,829858,1.414206,103.907934,PUNGGOL COAST BUS INTERCHANGE,1,SENTUL WALK,OneMap
2,Punggol Coast Bus Interchange,84 PUNGGOL WAY PUNGGOL COAST HAWKER CENTRE SIN...,829911,1.414553,103.908235,PUNGGOL COAST HAWKER CENTRE,84,PUNGGOL WAY,OneMap
3,White Cloud Cafe,88 PUNGGOL WAY PUNGGOL COAST MALL SINGAPORE 82...,829913,1.414711,103.910831,PUNGGOL COAST MALL,88,PUNGGOL WAY,OneMap
4,Nexus,101 NEW PUNGGOL ROAD PUNGGOL COAST MRT STATION...,828604,1.414927,103.910166,PUNGGOL COAST MRT STATION (NE18),101,NEW PUNGGOL ROAD,OneMap


In [17]:
df.tail()

,address,postal,lat,lon,building,block,road,source
988,70 PUNGGOL DRIVE MOE KINDERGARTEN @ WATERWAY S...,828802,1.399726,103.918346,MOE KINDERGARTEN @ WATERWAY,70,PUNGGOL DRIVE,OneMap
989,162 PUNGGOL CENTRAL PEARL STUDENT CARE CENTRE ...,820162,1.396113,103.914366,PEARL STUDENT CARE CENTRE,162,PUNGGOL CENTRAL,OneMap
990,176A EDGEFIELD PLAINS PUNGGOL NORTH FIRE POST ...,821176,1.398998,103.908146,PUNGGOL NORTH FIRE POST,176A,EDGEFIELD PLAINS,OneMap
991,83 PUNGGOL CENTRAL SHAW THEATRES WATERWAY POIN...,828761,1.406536,103.901988,SHAW THEATRES WATERWAY POINT,83,PUNGGOL CENTRAL,OneMap
992,11 NEW PUNGGOL ROAD SINGAPORE INSTITUTE OF TEC...,828616,1.414190,103.910623,SINGAPORE INSTITUTE OF TECHNOLOGY,11,NEW PUNGGOL ROAD,OneMap


In [23]:
df.to_csv("onemap_1000_with_place_names.csv", index=False)

from google.colab import files
files.download("onemap_1000_rows.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
import pandas as pd
from math import radians, sin, cos, sqrt, atan2
from difflib import SequenceMatcher

# Load data
osm = pd.read_csv("/content/osm_baseline.csv")
one = pd.read_csv("/content/onemap_1000_rows.csv")

# Distance function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

# Name similarity
def name_sim(a, b):
    if pd.isna(a) or pd.isna(b):
        return 0
    return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()

results = []
match_count = 0

for _, o in osm.iterrows():
    best_conf = 0

    for _, n in one.iterrows():
        # Name score
        ns = name_sim(o['Place Name'], n['building'])

        # Distance
        dist = haversine(o['lat'], o['lon'], n['lat'], n['lon'])

        # Location score
        if dist < 0.5:
            loc_score = 1.0
        elif dist < 2:
            loc_score = 0.7
        elif dist < 10:
            loc_score = 0.3
        else:
            loc_score = 0

        # Confidence
        conf = 0.6 * ns + 0.4 * loc_score

        if conf > best_conf:
            best_conf = conf

    # Status
    if best_conf >= 0.8:
        status = "OPEN"
        match_count += 1
    elif best_conf >= 0.5:
        status = "NEWLY OPEN"
    else:
        status = "PERMANENTLY CLOSED"

    results.append({
        "Place Name": o['Place Name'],
        "Confidence": best_conf,
        "Status": status
    })

df_result = pd.DataFrame(results)

print("Total Matches (OPEN):", match_count)
print(df_result.head())

Total Matches (OPEN): 0
       Place Name  Confidence              Status
0   Bonsai Garden    0.415385  PERMANENTLY CLOSED
1             NaN    0.280000  PERMANENTLY CLOSED
2  NTUC Fairprice    0.537143          NEWLY OPEN
3           Giant    0.480000  PERMANENTLY CLOSED
4  NTUC FairPrice    0.536364          NEWLY OPEN


In [30]:
import pandas as pd
from math import radians, cos, sin, asin, sqrt
from difflib import SequenceMatcher

# =========================
# 1. LOAD DATA
# =========================
osm = pd.read_csv("osm_baseline.csv")
tomtom = pd.read_csv("tomtom_singapore_poi.csv")

# =========================
# 2. STANDARDIZE COLUMNS
# =========================
osm = osm.rename(columns={
    'Latitude': 'lat',
    'Longitude': 'lon',
    'Place Name': 'name' # Corrected 'Name' to 'Place Name'
})

tomtom = tomtom.rename(columns={
    'latitude': 'lat',
    'longitude': 'lon',
    'name': 'name'
})

# =========================
# 3. CLEAN NAMES
# =========================
def clean_name(x):
    return str(x).lower().replace("'", "").replace("-", " ").strip()

osm['name_clean'] = osm['name'].apply(clean_name)
tomtom['name_clean'] = tomtom['name'].apply(clean_name)

# =========================
# 4. STRING SIMILARITY (NO LIB)
# =========================
def similarity(a, b):
    return int(SequenceMatcher(None, a, b).ratio() * 100)

# =========================
# 5. HAVERSINE DISTANCE
# =========================
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return 6371000 * c  # meters

# =========================
# 6. MATCHING
# =========================
matches = []

for i, o in osm.iterrows():
    best_match = None
    best_score = 0

    for j, t in tomtom.iterrows():
        dist = haversine(o['lat'], o['lon'], t['lat'], t['lon'])

        if dist <= 100:  # distance threshold
            score = similarity(o['name_clean'], t['name_clean'])

            if score > best_score:
                best_score = score
                best_match = {
                    'osm_name': o['name'],
                    'tomtom_name': t['name'],
                    'osm_lat': o['lat'],
                    'osm_lon': o['lon'],
                    'tomtom_lat': t['lat'],
                    'tomtom_lon': t['lon'],
                    'distance_m': round(dist, 2),
                    'name_score': score
                }

    if best_match and best_score >= 75:
        matches.append(best_match)

# =========================
# 7. SAVE RESULTS
# =========================
matches_df = pd.DataFrame(matches)

matches_df = matches_df.sort_values(
    by=['distance_m', 'name_score'],
    ascending=[True, False]
)

matches_df.to_csv("matched_poi.csv", index=False)

print(f"✅ Matching complete: {len(matches_df)} matches found")
print("📁 Output saved as: matched_poi.csv")

✅ Matching complete: 99 matches found
📁 Output saved as: matched_poi.csv


In [31]:
import pandas as pd
from math import radians, cos, sin, asin, sqrt
from difflib import SequenceMatcher

# =========================
# 1. LOAD DATA
# =========================
osm = pd.read_csv("osm_baseline.csv")
tomtom = pd.read_csv("tomtom_singapore_poi.csv")

# =========================
# 2. STANDARDIZE COLUMNS
# =========================
osm = osm.rename(columns={
    'Latitude': 'lat',
    'Longitude': 'lon',
    'Place Name': 'name'
})

tomtom = tomtom.rename(columns={
    'latitude': 'lat',
    'longitude': 'lon',
    'name': 'name'
})

# =========================
# 3. CLEAN NAMES
# =========================
def clean_name(x):
    return str(x).lower().replace("'", "").replace("-", " ").strip()

osm['name_clean'] = osm['name'].apply(clean_name)
tomtom['name_clean'] = tomtom['name'].apply(clean_name)

# =========================
# 4. STRING SIMILARITY
# =========================
def similarity(a, b):
    return int(SequenceMatcher(None, a, b).ratio() * 100)

# =========================
# 5. HAVERSINE DISTANCE
# =========================
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return 6371000 * c  # meters

# =========================
# 6. MATCHING
# =========================
matches = []
matched_tomtom_indices = set()

for i, o in osm.iterrows():
    best_match = None
    best_score = 0
    best_j = None

    for j, t in tomtom.iterrows():
        dist = haversine(o['lat'], o['lon'], t['lat'], t['lon'])

        if dist <= 100:
            score = similarity(o['name_clean'], t['name_clean'])

            if score > best_score:
                best_score = score
                best_match = t
                best_j = j

    if best_match is not None and best_score >= 75:
        matched_tomtom_indices.add(best_j)

# =========================
# 7. FIND UNMATCHED TOMTOM
# =========================
unmatched_tomtom = tomtom[~tomtom.index.isin(matched_tomtom_indices)]

# =========================
# 8. SAVE RESULTS
# =========================
unmatched_tomtom.to_csv("tomtom_not_in_osm.csv", index=False)

print(f"✅ Unmatched TomTom POIs: {len(unmatched_tomtom)}")
print("📁 Output saved as: tomtom_not_in_osm.csv")

✅ Unmatched TomTom POIs: 1293
📁 Output saved as: tomtom_not_in_osm.csv


In [32]:
import pandas as pd

df = pd.read_csv("tomtom_not_in_osm.csv")

print(df.head())   # shows first 5 rows

                 place_id                                      name  \
0  lx0bPzJIjCr1DZefcTMMAw                           Phbjv-Dormitory   
1  spRgvz4K_ub_1f18GaxAfg                         Greenhouse Bistro   
2  jtCxdxqXQRIq0qn0R06lPQ  Hi-Five Restaurant & Catering Point East   
3  kCT7I63Xdt9-a091PaVB3g                        Wow West Food Loft   
4  SNPGiWts5H7-PqBcrAmAcw           Allied Vending Hot Food Machine   

                     category       lat         lon  \
0              ['restaurant']  1.235298  103.616796   
1  ['american', 'restaurant']  1.315293  103.631750   
2    ['fusion', 'restaurant']  1.310741  103.630171   
3              ['restaurant']  1.308100  103.631550   
4              ['restaurant']  1.306700  103.664900   

                                    address  \
0                             Singapore, 63   
1          30 Tuas Bay Drive, Singapore, 63   
2      14 Tech Park Crescent, Singapore, 63   
3  2 Tuas South Street 2, Singapore, 637895   
4      

In [35]:
df.head(30)

,place_id,name,category,lat,lon,address,name_clean
0,lx0bPzJIjCr1DZefcTMMAw,Phbjv-Dormitory,['restaurant'],1.235298,103.616796,"Singapore, 63",phbjv dormitory
1,spRgvz4K_ub_1f18GaxAfg,Greenhouse Bistro,"['american', 'restaurant']",1.315293,103.631750,"30 Tuas Bay Drive, Singapore, 63",greenhouse bistro
2,jtCxdxqXQRIq0qn0R06lPQ,Hi-Five Restaurant & Catering Point East,"['fusion', 'restaurant']",1.310741,103.630171,"14 Tech Park Crescent, Singapore, 63",hi five restaurant & catering point east
3,kCT7I63Xdt9-a091PaVB3g,Wow West Food Loft,['restaurant'],1.308100,103.631550,"2 Tuas South Street 2, Singapore, 637895",wow west food loft
4,SNPGiWts5H7-PqBcrAmAcw,Allied Vending Hot Food Machine,['restaurant'],1.306700,103.664900,"2 Pioneer Sector 3, Singapore, 62",allied vending hot food machine
5,UFouH_0CLHN4OOaSP9-N6g,The Bay Cafe,"['fast food', 'restaurant']",1.251300,103.615400,"Tuas South Boulevard, Singapore, 63",the bay cafe
6,AMytUBwi5Up1GKFo94CgiQ,Liang Soon Paintwork Specialist Point East,['restaurant'],1.311365,103.657674,"Pioneer Road, Singapore, 62",liang soon paintwork specialist point east
7,vU8idJUNyrZPJ3A6Fs_rTg,San San Food House,['restaurant'],1.321900,103.633900,"Tuas Bay Drive, Singapore, 63",san san food house
8,OmQ9fglALKtyNLflK4npig,J & A Food Point East,"['fast food', 'restaurant']",1.294500,103.623100,"2 Tuas South Street 5, Singapore, 63",j & a food point east
9,acEf5nf0eyPLt-PwPZ583A,Hui Huang Roasted Delicious Pte.-Singapore,"['fast food', 'restaurant']",1.294500,103.668200,"Gul Road, Singapore, 627897",hui huang roasted delicious pte. singapore


In [38]:
from google.colab import files
files.download("tomtom_not_in_osm.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>